# Activation Heatmap Animation â€” Correct vs Hallucinated

Loads two groups of scans from MongoDB (separated by `ground_truth`) and renders
a side-by-side animated heatmap: **correct | hallucinated | absolute difference**.

Each frame is one scan snapshot. Y-axis = layer index (capture order), X-axis = zone / hidden-dim index.
The difference panel uses a hot colormap to highlight where activations diverge most.

## Cell 1 â€” Imports

In [ ]:
import sys, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from neuralsignal.backend.ns_backend import NSBackend
from neuralsignal.core.modules.neuralsignal_config import sdk_config

print(f"torch : {torch.__version__}")

## Cell 2 â€” Config

- `GROUND_TRUTH_CORRECT` / `GROUND_TRUTH_HALLUCINATED`: the `ground_truth` field values in MongoDB that distinguish the two groups.
- `N_SCANS`: number of frames in the animation (one scan per frame).
- `INTERVAL_MS`: milliseconds per frame.
- `mpl.rcParams["animation.embed_limit"]`: max embedded size in MB (default 20); raise if you see the size warning.

In [ ]:
import matplotlib as mpl

APPLICATION_NAME     = "redis"          # adjust
SUB_APPLICATION_NAME = "quora_duplicates_t5_large"  # adjust

GROUND_TRUTH_CORRECT      = "0"   # adjust per dataset
GROUND_TRUTH_HALLUCINATED = "1"   # adjust per dataset

N_SCANS     = 100
INTERVAL_MS = 300   # ms per animation frame
LAYER_STRIDE = 4    # show every Nth layer (reduces 553 to ~138 rows for visibility)

# Raise embed limit (MB) if animation exceeds the 20 MB default
mpl.rcParams["animation.embed_limit"] = 50

## Cell 3 â€” Connect to backend & load scan groups

In [ ]:
import tempfile

backend_cfg = sdk_config.get_backend_config()
backend_cfg["scan_cache_directory"] = tempfile.gettempdir()

cfg = {
    "application_name": APPLICATION_NAME,
    "sub_application_name": SUB_APPLICATION_NAME,
    "backend_config": backend_cfg,
}
be = NSBackend(cfg)


def load_group(be, ground_truth_value, n):
    query = {"ground_truth": ground_truth_value}
    cursor = be.query(query).limit(n)
    scans = []
    for doc in cursor:
        try:
            scans.append(be.deserialize_scan(doc))
        except Exception as e:
            print(f"  skipping {doc.get('_id')}: {e}")
    return scans


print("Loading correct group ...")
correct_scans = load_group(be, GROUND_TRUTH_CORRECT, N_SCANS)
print(f"  loaded {len(correct_scans)} scans")

print("Loading hallucinated group ...")
halu_scans = load_group(be, GROUND_TRUTH_HALLUCINATED, N_SCANS)
print(f"  loaded {len(halu_scans)} scans")

## Cell 4 â€” Convert a scan to a 2-D heatmap array

Output shape: `(num_layers, zone_size)`.  
Each row = mean activation across sequence positions for that layer.

In [ ]:
def scan_to_heatmap(scan):
    """Return (num_layers, zone_size) float32 numpy array."""
    rows = []
    for lid in scan["layer_order"]:
        tensor = scan["outputs"].get(lid)
        if tensor is None:
            continue
        t = tensor.float().cpu()
        if t.dim() == 1:
            rows.append(t.numpy())              # already zoned: (zone_size,)
        elif t.dim() == 2:
            rows.append(t.mean(dim=0).numpy())  # (seq_len, hidden) -> (hidden,)
        # skip tensors with unexpected dims
    if not rows:
        return None
    max_len = max(r.shape[0] for r in rows)
    padded = [np.pad(r, (0, max_len - r.shape[0])) for r in rows]
    return np.array(padded, dtype=np.float32)   # (num_layers, zone_size)


def _diff(a, b):
    """Absolute difference, zero-padded to same shape if needed."""
    if a.shape == b.shape:
        return np.abs(a - b)
    rows = max(a.shape[0], b.shape[0])
    cols = max(a.shape[1], b.shape[1])
    pa = np.zeros((rows, cols), dtype=np.float32)
    pb = np.zeros((rows, cols), dtype=np.float32)
    pa[:a.shape[0], :a.shape[1]] = a
    pb[:b.shape[0], :b.shape[1]] = b
    return np.abs(pa - pb)


# Smoke-test on first scan
sample = scan_to_heatmap(correct_scans[0])
print(f"Sample heatmap shape: {sample.shape}  (num_layers x zone_size)")

## Cell 5 â€” Pre-compute heatmaps & global colour scale

In [ ]:
correct_heatmaps = [scan_to_heatmap(s) for s in correct_scans]
halu_heatmaps    = [scan_to_heatmap(s) for s in halu_scans]

# Drop any scans that failed conversion
correct_heatmaps = [(h, s) for h, s in zip(correct_heatmaps, correct_scans) if h is not None]
halu_heatmaps    = [(h, s) for h, s in zip(halu_heatmaps,    halu_scans)    if h is not None]

n_frames = min(len(correct_heatmaps), len(halu_heatmaps), N_SCANS)
print(f"Animating {n_frames} frames")

# Subsample layers -- 553 rows in a small figure is visually compressed
correct_heatmaps = [(h[::LAYER_STRIDE], s) for h, s in correct_heatmaps]
halu_heatmaps    = [(h[::LAYER_STRIDE], s) for h, s in halu_heatmaps]
print(f"Heatmap shape after stride-{LAYER_STRIDE}: {correct_heatmaps[0][0].shape}")

# Shared colour scale across all frames and both groups
all_vals = np.concatenate([h.ravel() for h, _ in correct_heatmaps + halu_heatmaps])
vmin, vmax = float(np.percentile(all_vals, 2)), float(np.percentile(all_vals, 98))
print(f"Colour scale: [{vmin:.4f}, {vmax:.4f}]  (2nd-98th percentile)")

# Pre-compute all diff heatmaps and a GLOBAL diff scale.
# Per-frame scaling caused the diff panel to appear blank: frames with small
# diffs showed all-black because hot-colormap(0) = black and set_clim was
# anchored to each frame's own max.
diff_heatmaps = [
    _diff(correct_heatmaps[i][0], halu_heatmaps[i][0])
    for i in range(n_frames)
]
diff_vals = np.concatenate([d.ravel() for d in diff_heatmaps])
diff_vmax = float(np.percentile(diff_vals, 98)) or 1.0
print(f"Diff scale: [0, {diff_vmax:.4f}]  (98th percentile across all frames)")

## Cell 6 â€” Build & display animation

Three panels per frame:
1. **Correct** â€” ground_truth = GROUND_TRUTH_CORRECT
2. **Hallucinated** â€” ground_truth = GROUND_TRUTH_HALLUCINATED
3. **|Difference|** â€” absolute activation difference, hot colormap

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fig.suptitle(
    f"Activation Heatmaps -- Correct (gt={GROUND_TRUTH_CORRECT}) vs "
    f"Hallucinated (gt={GROUND_TRUTH_HALLUCINATED})",
    fontsize=13,
)

panel_titles = ["Correct", "Hallucinated", "| Difference |"]
for ax, title in zip(axes, panel_titles):
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Zone / Hidden-dim index")
    ax.set_ylabel(f"Layer index (every {LAYER_STRIDE}th)")

# initial frame
h_c, s_c = correct_heatmaps[0]
h_h, s_h = halu_heatmaps[0]
h_d       = diff_heatmaps[0]

im_c = axes[0].imshow(h_c, aspect="auto", vmin=vmin,     vmax=vmax,     cmap="viridis", origin="upper")
im_h = axes[1].imshow(h_h, aspect="auto", vmin=vmin,     vmax=vmax,     cmap="viridis", origin="upper")
im_d = axes[2].imshow(h_d, aspect="auto", vmin=0,        vmax=diff_vmax, cmap="hot",    origin="upper")

fig.colorbar(im_c, ax=axes[0], fraction=0.046, pad=0.04)
fig.colorbar(im_h, ax=axes[1], fraction=0.046, pad=0.04)
fig.colorbar(im_d, ax=axes[2], fraction=0.046, pad=0.04)

frame_label = fig.text(
    0.5, 0.01,
    f"Frame 1/{n_frames}",
    ha="center", fontsize=9, style="italic",
)

def update(i):
    h_c, s_c = correct_heatmaps[i]
    h_h, s_h = halu_heatmaps[i]
    h_d       = diff_heatmaps[i]

    im_c.set_data(h_c)
    im_h.set_data(h_h)
    im_d.set_data(h_d)
    # clim is fixed to global diff_vmax -- no per-frame rescaling

    inp_c = str(s_c.get("input", ""))[:80]
    inp_h = str(s_h.get("input", ""))[:80]
    frame_label.set_text(
        f"Frame {i+1}/{n_frames}  |  "
        f"Correct: {inp_c}...  |  "
        f"Halu: {inp_h}..."
    )

ani = animation.FuncAnimation(
    fig, update, frames=n_frames, interval=INTERVAL_MS, blit=False
)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
HTML(ani.to_jshtml())

## Cell 7 â€” Export to file (optional)

Requires `ffmpeg` on PATH for `.mp4`, or `pillow` for `.gif`.

In [ ]:
# Uncomment one of the lines below to export:
# ani.save("activation_animation.mp4", writer="ffmpeg", fps=4, dpi=120)
# ani.save("activation_animation.gif",  writer="pillow",  fps=4)